# Nika on Colab

Train the ~30M model on a bigger slice of FineWeb-Edu.

**Before running:** Runtime -> Change runtime type -> GPU (T4).

Checkpoints go to Google Drive, so a disconnect costs at most one eval interval.
Rerun every cell after a disconnect: the training cell has `--resume`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.is_bf16_supported())

In [ ]:
# clone (or update) the repo and work from its root, so `python -m scripts.x` imports work
import os
REPO = "https://github.com/dat999zx/nika.git"
if not os.path.exists("/content/nika"):
    !git clone $REPO /content/nika
%cd /content/nika
!git pull
!pip -q install datasets

In [ ]:
# Drive holds the checkpoints: /content is wiped when the session ends
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/nika/nika30m.pt'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
print(CKPT)

In [ ]:
# ~2 GB of text -> about 600M tokens at 3.4 chars/token. Takes a few minutes.
# Skip this cell if data/tiny.txt already exists from an earlier run in this session.
!python -m scripts.download_data --mb 2000

In [ ]:
# tokenize with the committed 4096-vocab tokenizer (data/tokenizer.txt), one process per core
!python -m scripts.encode_data

In [ ]:
# ~30M params: n_embed 512, 6 layers, 8 heads (head_size 64), context 256.
# --resume picks up from the _last.pt checkpoint if there is one, so rerun this cell
# after a disconnect. Chinchilla-ish budget: about 600M tokens
# = max_iters * batch_size * block_size = 20000 * 48 * 256 ~ 245M, so run it more than once
# (raise --max-iters and rerun with --resume) if you want the full budget.
!python -m scripts.train \
    --n-embed 512 --n-layer 6 --n-head 8 --block-size 256 \
    --batch-size 48 --max-iters 20000 --lr 6e-4 \
    --eval-interval 500 --eval-iters 40 --warmup-iters 500 \
    --checkpoint $CKPT --resume

In [ ]:
# the report, with the loss table and curve
from IPython.display import Markdown, display
display(Markdown(open(CKPT.replace('.pt', '_report.md')).read()))

In [ ]:
!python -m scripts.generate --checkpoint $CKPT --prompt "Photosynthesis is the process by which plants" --tokens 150 --temperature 0.8